# 01 – Nettoyage des données

But : partir des 4 fichiers bruts (`data/raw/`) et produire des fichiers propres (`data/processed/`)
prêts pour l'exploration et le modèle.

On vérifie, pour chaque fichier :
1. **En-têtes** : noms de colonnes cohérents (snake_case) et colonnes attendues présentes
2. **Format** : dates en vraies dates, scores en entiers, booléens en vrais booléens, textes sans espaces parasites
3. **Valeurs vides** : où elles sont et si elles sont graves
4. **Doublons** : exacts, ou sur la clé du match
5. **Valeurs erronées** : scores négatifs, équipe contre elle-même, minute impossible, vainqueur qui n'a pas joué…
6. **Cohérence entre fichiers** : chaque but / tir au but correspond bien à un match

Les fonctions de nettoyage sont dans `src/nettoyage.py` (testées dans `tests/test_nettoyage.py`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from download import download_csv_files
from nettoyage import (
    nettoyer_former_names,
    nettoyer_goalscorers,
    nettoyer_results,
    nettoyer_shootouts,
)

DOSSIER_BRUT = RACINE / "data" / "raw"
DOSSIER_PROPRE = RACINE / "data" / "processed"
FICHIERS = ["results", "goalscorers", "shootouts", "former_names"]

## 1. Récupérer les données brutes
Si les fichiers ne sont pas là, on réutilise le script `src/download.py` de l'équipe.

In [2]:
manquants = [f"{nom}.csv" for nom in FICHIERS if not (DOSSIER_BRUT / f"{nom}.csv").exists()]
if manquants:
    download_csv_files("martj42", "international_results", manquants,
                       folder=str(DOSSIER_BRUT), branch="master")

brut = {nom: pd.read_csv(DOSSIER_BRUT / f"{nom}.csv", encoding="utf-8") for nom in FICHIERS}
{nom: df.shape for nom, df in brut.items()}

{'results': (49940, 9),
 'goalscorers': (47914, 9),
 'shootouts': (700, 5),
 'former_names': (36, 4)}

## 2. Diagnostic avant nettoyage
Une petite fonction qui résume les défauts d'un tableau.

In [3]:
def diagnostic(df: pd.DataFrame) -> pd.DataFrame:
    """Une ligne par colonne : type, nb de vides, % de vides, nb de valeurs différentes."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "vides": df.isna().sum(),
        "% vides": (df.isna().mean() * 100).round(1),
        "valeurs différentes": df.nunique(),
    })

for nom, df in brut.items():
    print(f"=== {nom} : {df.shape[0]} lignes, {df.duplicated().sum()} doublons exacts")
    display(diagnostic(df))

=== results : 49940 lignes, 0 doublons exacts


,type,vides,% vides,valeurs différentes
date,str,0,0.0,16538
home_team,str,0,0.0,328
away_team,str,0,0.0,322
home_score,int64,0,0.0,26
away_score,int64,0,0.0,22
tournament,str,0,0.0,225
city,str,0,0.0,2098
country,str,0,0.0,269
neutral,bool,0,0.0,2


=== goalscorers : 47914 lignes, 79 doublons exacts


,type,vides,% vides,valeurs différentes
date,str,0,0.0,4761
home_team,str,0,0.0,220
away_team,str,0,0.0,220
team,str,0,0.0,220
scorer,str,44,0.1,15343
minute,float64,254,0.5,121
own_goal,bool,0,0.0,2
penalty,bool,0,0.0,2
assist,str,45838,95.7,1340


=== shootouts : 700 lignes, 0 doublons exacts


,type,vides,% vides,valeurs différentes
date,str,0,0.0,608
home_team,str,0,0.0,189
away_team,str,0,0.0,200
winner,str,0,0.0,185
first_shooter,str,431,61.6,98


=== former_names : 36 lignes, 0 doublons exacts


,type,vides,% vides,valeurs différentes
current,str,0,0.0,28
former,str,0,0.0,36
start_date,str,0,0.0,36
end_date,str,0,0.0,36


**Ce qu'on remarque :**
- Les dates sont lues comme du **texte**, pas comme des dates → à convertir.
- `goalscorers` : `assist` est presque toujours vide (normal, info rarement connue), `minute` et `scorer` ont quelques vides.
- `shootouts` : `first_shooter` souvent vide (info absente des vieux matchs).
- `goalscorers` a des **doublons exacts** → à regarder de près avant de supprimer.

In [4]:
g = brut["goalscorers"]
g[g.duplicated(keep=False)].head(6)

,date,home_team,away_team,team,scorer,minute,own_goal,penalty,assist
4059,1963-11-26,Ghana,Ethiopia,Ghana,Edward Acquah,NaN,False,False,NaN
4060,1963-11-26,Ghana,Ethiopia,Ghana,Edward Acquah,NaN,False,False,NaN
4063,1963-11-28,Ethiopia,Tunisia,Ethiopia,Mengistu Worku,NaN,False,False,NaN
4064,1963-11-28,Ethiopia,Tunisia,Ethiopia,Mengistu Worku,NaN,False,False,NaN
4067,1963-11-28,Nigeria,Sudan,Sudan,Nasr Eddin Abbas,NaN,False,False,NaN
4068,1963-11-28,Nigeria,Sudan,Sudan,Nasr Eddin Abbas,NaN,False,False,NaN


Ces « doublons » ont tous une **minute vide** : c'est le même joueur qui a marqué **deux fois**
dans le même match, mais on ne sait pas à quelle minute. Ce sont de **vrais buts** :
les supprimer fausserait le nombre de buts. **On les garde.**

In [5]:
r = brut["results"]
cle = ["date", "home_team", "away_team"]
print("Doublons sur la clé (date, domicile, extérieur) :", r.duplicated(cle).sum())
r[r.duplicated(cle, keep=False)]

Doublons sur la clé (date, domicile, extérieur) : 1


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
9648,1974-02-17,Tahiti,New Caledonia,2,1,Friendly,Papeete,Tahiti,False
9649,1974-02-17,Tahiti,New Caledonia,1,2,Friendly,Papeete,Tahiti,False


Deux matchs Tahiti – Nouvelle-Calédonie le même jour avec des scores différents.
On ne peut pas savoir lequel est faux (ou si deux matchs ont vraiment eu lieu) : on les **garde**,
mais on le note. Ce n'est pas un doublon exact, donc `drop_duplicates()` ne les touche pas.

## 3. Nettoyage

In [6]:
propre = {
    "results": nettoyer_results(brut["results"]),
    "goalscorers": nettoyer_goalscorers(brut["goalscorers"]),
    "shootouts": nettoyer_shootouts(brut["shootouts"]),
    "former_names": nettoyer_former_names(brut["former_names"]),
}

bilan = pd.DataFrame({
    "lignes avant": {n: len(brut[n]) for n in FICHIERS},
    "lignes après": {n: len(propre[n]) for n in FICHIERS},
})
bilan["retirées"] = bilan["lignes avant"] - bilan["lignes après"]
bilan

,lignes avant,lignes après,retirées
results,49940,49940,0
goalscorers,47914,47914,0
shootouts,700,700,0
former_names,36,36,0


In [7]:
for nom, df in propre.items():
    print(f"=== {nom}")
    display(diagnostic(df))

=== results


,type,vides,% vides,valeurs différentes
date,datetime64[us],0,0.0,16538
home_team,string,0,0.0,328
away_team,string,0,0.0,322
home_score,Int64,0,0.0,26
away_score,Int64,0,0.0,22
tournament,string,0,0.0,225
city,string,0,0.0,2098
country,string,0,0.0,269
neutral,bool,0,0.0,2


=== goalscorers


,type,vides,% vides,valeurs différentes
date,datetime64[us],0,0.0,4761
home_team,string,0,0.0,220
away_team,string,0,0.0,220
team,string,0,0.0,220
scorer,string,44,0.1,15343
minute,Int64,254,0.5,121
own_goal,bool,0,0.0,2
penalty,bool,0,0.0,2
assist,string,45838,95.7,1340


=== shootouts


,type,vides,% vides,valeurs différentes
date,datetime64[us],0,0.0,608
home_team,string,0,0.0,189
away_team,string,0,0.0,200
winner,string,0,0.0,185
first_shooter,string,431,61.6,98


=== former_names


,type,vides,% vides,valeurs différentes
current,string,0,0.0,28
former,string,0,0.0,36
start_date,datetime64[us],0,0.0,36
end_date,datetime64[us],0,0.0,36


Les dates sont maintenant de type `datetime64`, les scores `Int64`, les booléens `bool`.

Aucune ligne n'est retirée : la source est déjà bien tenue (pas de score négatif, pas d'équipe
contre elle-même, pas de date illisible). Le nettoyage reste utile : il **vérifie** tout ça,
corrige les **types**, et protège le projet si un futur téléchargement contient des erreurs.

## 4. Valeurs extrêmes : erreurs ou vrais résultats ?

In [8]:
res = propre["results"]
res[(res["home_score"] >= 15) | (res["away_score"] >= 15)].sort_values("home_score", ascending=False).head(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
25432,2001-04-11,Australia,American Samoa,31,0,FIFA World Cup qualification,Coffs Harbour,Australia,False
8551,1971-09-13,Tahiti,Cook Islands,30,0,South Pacific Games,Papeete,Tahiti,False
11919,1979-08-30,Fiji,Kiribati,24,0,South Pacific Games,Nausori,Fiji,False
25429,2001-04-09,Australia,Tonga,22,0,FIFA World Cup qualification,Coffs Harbour,Australia,False
37074,2013-06-24,Provence,Tibet,22,0,"International Tournament of Peoples, Cultures ...",Marseille,France,False


Les gros scores (ex. Australie 31–0 Samoa américaines en 2001) sont **réels** : on les garde.
Pour le modèle, il faudra peut-être limiter leur poids, mais ce n'est pas du nettoyage.

À noter : `results` contient aussi des équipes **non membres de la FIFA** (ex. Provence, Tibet).
Ce ne sont pas des erreurs, mais il faudra sûrement les filtrer pour prédire la Coupe du monde.

In [9]:
buts = propre["goalscorers"]
print("Minutes des buts : de", buts["minute"].min(), "à", buts["minute"].max())
print("Minutes inconnues :", buts["minute"].isna().sum())

Minutes des buts : de 1 à 122
Minutes inconnues : 254


## 5. Cohérence entre fichiers

In [10]:
cle = ["date", "home_team", "away_team"]

buts_sans_match = buts.merge(res[cle], on=cle, how="left", indicator=True).query("_merge == 'left_only'")
print("Buts sans match correspondant :", len(buts_sans_match))

tab = propre["shootouts"].merge(res[cle + ["home_score", "away_score"]], on=cle, how="left", indicator=True)
tab_sans_match = tab[tab["_merge"] == "left_only"]
print("Tirs au but sans match correspondant :", len(tab_sans_match))
display(tab_sans_match[cle + ["winner"]])

Buts sans match correspondant : 0
Tirs au but sans match correspondant : 1


,date,home_team,away_team,winner
429,2011-06-29,Saare County,Åland Islands,Åland Islands


Saare County – Åland Islands : ce match n'existe pas dans `results`. Sans le score du match,
cette ligne ne sert à rien : on la **retire**.

In [11]:
tab_avec_match = tab[tab["_merge"] == "both"]
pas_nul = tab_avec_match[tab_avec_match["home_score"] != tab_avec_match["away_score"]]
print("Tirs au but après un match NON nul :", len(pas_nul))
pas_nul[cle + ["home_score", "away_score", "winner"]].head()

Tirs au but après un match NON nul : 37


,date,home_team,away_team,home_score,away_score,winner
7,1973-04-21,Senegal,Ghana,1,0,Ghana
22,1974-11-22,Libya,Tunisia,1,0,Tunisia
25,1975-07-13,Morocco,Ghana,2,0,Morocco
28,1975-12-23,Libya,Syria,1,0,Syria
33,1977-06-26,Zambia,Algeria,2,0,Zambia


Ce n'est pas une erreur : ce sont souvent des **matchs aller-retour** (le score cumulé des deux matchs
est à égalité, puis tirs au but). On les **garde**.

In [12]:
propre["shootouts"] = propre["shootouts"].merge(res[cle], on=cle, how="inner")
print("Tirs au but gardés :", len(propre["shootouts"]))

Tirs au but gardés : 699


## 6. Sauvegarde dans `data/processed/`

In [13]:
DOSSIER_PROPRE.mkdir(parents=True, exist_ok=True)
for nom, df in propre.items():
    chemin = DOSSIER_PROPRE / f"{nom}.csv"
    df.to_csv(chemin, index=False, date_format="%Y-%m-%d")
    print(f"{nom:13s} → {chemin.relative_to(RACINE)} ({len(df)} lignes)")

results       → data\processed\results.csv (49940 lignes)


goalscorers   → data\processed\goalscorers.csv (47914 lignes)
shootouts     → data\processed\shootouts.csv (699 lignes)
former_names  → data\processed\former_names.csv (36 lignes)


## Résumé

| Problème | Décision |
|---|---|
| Dates en texte | converties en vraies dates |
| Espaces / textes vides | supprimés / transformés en valeurs manquantes |
| En-têtes | normalisés + vérification des colonnes attendues |
| Doublons exacts dans `goalscorers` | **gardés** (vrais buts, minute inconnue) |
| Doublon de clé Tahiti 1974 | gardé, signalé |
| Scores négatifs, équipe contre elle-même | retirés (aucun trouvé ici) |
| Minute hors 1–130 | mise à « inconnue » |
| Tirs au but sans match | retirés (1 ligne) |
| Tirs au but après un match non nul | gardés (matchs aller-retour) |
| Équipes non FIFA (Provence, Tibet…) | gardées, à filtrer plus tard pour le modèle |

**Prochaine étape :** notebook d'exploration (`02_exploration.ipynb`) à partir de `data/processed/`.